# Layer 3: Data Cleaning and Transformation Notebook

## Industrial Equipment Performance & Maintenance Intelligence Analytics Platform

**Dataset:** AI4I 2020 Predictive Maintenance Dataset  
**Database:** `data/database/maintenance.db`  
**Output Tables:** `machine_clean`, `machine_analytics`

This notebook performs a professional data quality review, transformation, and database load for maintenance analytics. Every cleaning action is documented with a decision, reason, and business impact.

## Section 1: Business Context

Industrial maintenance analytics depends on trustworthy operational data. Sensor readings, equipment attributes, and failure labels are used to identify reliability patterns, calculate KPIs, prioritize maintenance, and train predictive models. If the data is inaccurate, incomplete, duplicated, or inconsistent, the platform can produce misleading failure rates, incorrect risk rankings, and poor maintenance recommendations.

### Why Data Quality Matters for Maintenance Analytics

- Maintenance decisions are operationally expensive and can affect production uptime.
- Failure labels are used as the target for predictive maintenance modeling.
- Sensor values such as torque, speed, temperature, and tool wear drive root-cause analysis.
- Power BI dashboards need stable, validated fields so leaders can trust KPIs.

### Risks of Poor-Quality Operational Data

- False failure patterns can trigger unnecessary preventive maintenance.
- Missing or invalid sensor values can weaken machine learning models.
- Duplicate records can inflate failure counts and distort failure rates.
- Incorrect data types can break SQL calculations and Power BI measures.
- Inconsistent failure labels can reduce confidence in maintenance intelligence.

### Cleaning Process Objectives

- Confirm the dataset follows the expected AI4I 2020 schema.
- Detect missing values, duplicates, invalid values, and type issues.
- Preserve valid operational extremes instead of automatically removing outliers.
- Engineer business-ready maintenance features.
- Create final clean and analytics-ready tables in SQLite.

In [1]:
from __future__ import annotations

import math
import sqlite3
import sys
from datetime import datetime, timezone
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 120)
sns.set_theme(style="whitegrid")

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DATA_PATH = PROJECT_ROOT / "data" / "raw" / "ai4i2020.csv"
DATABASE_PATH = PROJECT_ROOT / "data" / "database" / "maintenance.db"
DDL_PATH = PROJECT_ROOT / "sql" / "ddl" / "create_tables.sql"

print(f"Project root: {PROJECT_ROOT}")
print(f"Dataset path: {DATA_PATH}")
print(f"Database path: {DATABASE_PATH}")

Project root: C:\Users\Lavanya\Documents\Data analytics\Industrial Equipment Performance & Maintenance Intelligence Analytics Platform
Dataset path: C:\Users\Lavanya\Documents\Data analytics\Industrial Equipment Performance & Maintenance Intelligence Analytics Platform\data\raw\ai4i2020.csv
Database path: C:\Users\Lavanya\Documents\Data analytics\Industrial Equipment Performance & Maintenance Intelligence Analytics Platform\data\database\maintenance.db


In [2]:
EXPECTED_COLUMNS = [
    "UDI",
    "Product ID",
    "Type",
    "Air temperature [K]",
    "Process temperature [K]",
    "Rotational speed [rpm]",
    "Torque [Nm]",
    "Tool wear [min]",
    "Machine failure",
    "TWF",
    "HDF",
    "PWF",
    "OSF",
    "RNF",
]

COLUMN_RENAME_MAP = {
    "UDI": "observation_id",
    "Product ID": "product_id",
    "Type": "machine_type",
    "Air temperature [K]": "air_temperature_k",
    "Process temperature [K]": "process_temperature_k",
    "Rotational speed [rpm]": "rotational_speed_rpm",
    "Torque [Nm]": "torque_nm",
    "Tool wear [min]": "tool_wear_min",
    "Machine failure": "machine_failure",
    "TWF": "twf",
    "HDF": "hdf",
    "PWF": "pwf",
    "OSF": "osf",
    "RNF": "rnf",
}

NUMERIC_SOURCE_COLUMNS = [
    "UDI",
    "Air temperature [K]",
    "Process temperature [K]",
    "Rotational speed [rpm]",
    "Torque [Nm]",
    "Tool wear [min]",
    "Machine failure",
    "TWF",
    "HDF",
    "PWF",
    "OSF",
    "RNF",
]

INTEGER_CLEAN_COLUMNS = [
    "observation_id",
    "rotational_speed_rpm",
    "tool_wear_min",
    "machine_failure",
    "twf",
    "hdf",
    "pwf",
    "osf",
    "rnf",
]

FLOAT_CLEAN_COLUMNS = ["air_temperature_k", "process_temperature_k", "torque_nm"]
BINARY_CLEAN_COLUMNS = ["machine_failure", "twf", "hdf", "pwf", "osf", "rnf"]
SENSOR_COLUMNS = ["air_temperature_k", "process_temperature_k", "rotational_speed_rpm", "torque_nm", "tool_wear_min"]

quality_actions: list[dict[str, str]] = []

def record_action(section: str, issue: str, decision: str, reason: str, business_impact: str) -> None:
    quality_actions.append(
        {
            "Section": section,
            "Issue": issue,
            "Decision": decision,
            "Reason": reason,
            "Business Impact": business_impact,
        }
    )

def action_table(section: str) -> pd.DataFrame:
    rows = [row for row in quality_actions if row["Section"] == section]
    return pd.DataFrame(rows)

def require_dataset(path: Path) -> None:
    if not path.exists():
        raise FileNotFoundError(
            f"Dataset not found at {path}. Place the AI4I 2020 CSV file in data/raw and name it ai4i2020.csv, "
            "or update DATA_PATH in the first configuration cell."
        )
    if path.suffix.lower() != ".csv":
        raise ValueError(f"Expected a CSV file. Received: {path.name}")
    if path.stat().st_size == 0:
        raise ValueError(f"Dataset file is empty: {path}")

## Section 2: Dataset Overview

This section confirms the dataset structure before any cleaning or transformation occurs. The objective is to understand the row count, available fields, initial data types, and visible data quality observations.

In [3]:
require_dataset(DATA_PATH)
raw_df = pd.read_csv(DATA_PATH)

print(f"Dataset shape: {raw_df.shape[0]:,} rows x {raw_df.shape[1]:,} columns")
display(raw_df.head())

Dataset shape: 10,000 rows x 14 columns


,UDI,Product ID,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Machine failure,TWF,HDF,PWF,OSF,RNF
0,1,M14860,M,298.1,308.6,1551,42.8,0,0,0,0,0,0,0
1,2,L47181,L,298.2,308.7,1408,46.3,3,0,0,0,0,0,0
2,3,L47182,L,298.1,308.5,1498,49.4,5,0,0,0,0,0,0
3,4,L47183,L,298.2,308.6,1433,39.5,7,0,0,0,0,0,0
4,5,L47184,L,298.2,308.7,1408,40.0,9,0,0,0,0,0,0


In [4]:
overview = pd.DataFrame(
    {
        "column": raw_df.columns,
        "dtype": [str(dtype) for dtype in raw_df.dtypes],
        "non_null_count": raw_df.notna().sum().values,
        "unique_values": raw_df.nunique(dropna=True).values,
    }
)
display(overview)

,column,dtype,non_null_count,unique_values
0,UDI,int64,10000,10000
1,Product ID,object,10000,10000
2,Type,object,10000,3
3,Air temperature [K],float64,10000,93
4,Process temperature [K],float64,10000,82
5,Rotational speed [rpm],int64,10000,941
6,Torque [Nm],float64,10000,577
7,Tool wear [min],int64,10000,246
8,Machine failure,int64,10000,2
9,TWF,int64,10000,2


In [5]:
missing_columns = [column for column in EXPECTED_COLUMNS if column not in raw_df.columns]
unexpected_columns = [column for column in raw_df.columns if column not in EXPECTED_COLUMNS]
column_order_matches = list(raw_df.columns) == EXPECTED_COLUMNS

schema_summary = pd.DataFrame(
    [
        {"check": "Expected columns present", "result": len(missing_columns) == 0, "details": missing_columns},
        {"check": "No unexpected columns", "result": len(unexpected_columns) == 0, "details": unexpected_columns},
        {"check": "Expected column order", "result": column_order_matches, "details": "Matches" if column_order_matches else "Review order"},
    ]
)
display(schema_summary)

if missing_columns or unexpected_columns:
    raise ValueError(f"Schema validation failed. Missing: {missing_columns}. Unexpected: {unexpected_columns}.")

record_action(
Dataset Overview",
Dataset schema validation",
Proceed with AI4I expected schema after confirming all required fields are present.",
The AI4I fields are required for sensor analysis, failure labeling, feature engineering, and SQL table loading.",
Ensures downstream maintenance KPIs and Power BI fields are built on the correct operational data structure.",
)
display(action_table("Dataset Overview"))

SyntaxError: unterminated string literal (detected at line 18) (2733450760.py, line 18)

### Initial Observations

- The dataset is expected to contain one row per machine operating observation.
- `Machine failure` is the primary target variable for predictive maintenance analysis.
- `TWF`, `HDF`, `PWF`, `OSF`, and `RNF` provide failure category signals.
- Sensor and operating measurements are required for root-cause analytics and engineered maintenance features.

## Section 3: Missing Value Assessment

Missing values in maintenance analytics can hide failure patterns, weaken KPI accuracy, and introduce bias into predictive models. This section detects missing values and documents the recommended treatment.

In [ ]:
missing_summary = pd.DataFrame(
    {
        "column": raw_df.columns,
        "missing_count": raw_df.isna().sum().values,
        "missing_percent": (raw_df.isna().mean().values * 100).round(4),
    }
).sort_values("missing_count", ascending=False)

display(missing_summary)

In [ ]:
total_missing = int(raw_df.isna().sum().sum())

if total_missing == 0:
    record_action(
        "Missing Value Assessment",
        "No missing values detected",
        "No imputation, deletion, or replacement is applied.",
        "All required sensor, operating, and failure fields are complete.",
        "Preserves the full dataset while avoiding unnecessary assumptions that could distort maintenance KPIs or model features.",
    )
else:
    columns_with_missing = missing_summary.loc[missing_summary["missing_count"] > 0, "column"].tolist()
    record_action(
        "Missing Value Assessment",
        f"Missing values detected in {columns_with_missing}",
        "Stop the cleaning process and resolve missing values before database loading.",
        "Maintenance analytics requires complete failure labels and sensor measurements; automatic imputation could create false operating conditions.",
        "Prevents unreliable failure rates, distorted feature engineering, and misleading Power BI dashboard metrics.",
    )

display(action_table("Missing Value Assessment"))

if total_missing > 0:
    raise ValueError("Missing values must be resolved before continuing with Layer 3 cleaning.")

## Section 4: Duplicate Analysis

Duplicate records can overstate failure counts, bias failure rates, and distort machine reliability scores. This section checks full-row duplicates and business-key duplicates.

In [ ]:
duplicate_summary = pd.DataFrame(
    [
        {"duplicate_check": "Full row duplicates", "duplicate_count": int(raw_df.duplicated().sum())},
        {"duplicate_check": "Duplicate UDI", "duplicate_count": int(raw_df.duplicated(subset=["UDI"]).sum())},
        {"duplicate_check": "Duplicate Product ID", "duplicate_count": int(raw_df.duplicated(subset=["Product ID"]).sum())},
    ]
)

display(duplicate_summary)

In [ ]:
duplicate_issue_count = int(duplicate_summary["duplicate_count"].sum())

if duplicate_issue_count == 0:
    record_action(
        "Duplicate Analysis",
        "No duplicate rows or duplicate business keys detected",
        "Keep all records.",
        "Each observation appears unique by full row, UDI, and Product ID.",
        "Prevents accidental loss of valid observations and preserves accurate failure-rate denominators.",
    )
else:
    record_action(
        "Duplicate Analysis",
        "Duplicate records or duplicate business keys detected",
        "Stop the process and review duplicates before loading the database.",
        "Automatic duplicate removal may remove valid repeated operating conditions or hide source-system issues.",
        "Avoids inflated failure counts, distorted MTBF proxy, and incorrect maintenance burden calculations.",
    )

display(action_table("Duplicate Analysis"))

if duplicate_issue_count > 0:
    duplicate_examples = raw_df.loc[raw_df.duplicated(keep=False) | raw_df.duplicated(subset=["UDI"], keep=False)].head(10)
    display(duplicate_examples)
    raise ValueError("Duplicate issues must be reviewed before continuing with Layer 3 cleaning.")

## Section 5: Data Type Validation

This section validates numeric fields, categorical fields, and failure indicators. The goal is to ensure SQL calculations, visualizations, and model features use reliable data types.

In [ ]:
type_validation_rows = []

for column in NUMERIC_SOURCE_COLUMNS:
    converted = pd.to_numeric(raw_df[column], errors="coerce")
    invalid_count = int(converted.isna().sum())
    type_validation_rows.append(
        {
            "column": column,
            "expected_type": "numeric",
            "invalid_count": invalid_count,
            "validation_status": "Pass" if invalid_count == 0 else "Fail",
        }
    )

machine_type_values = sorted(raw_df["Type"].astype(str).str.strip().unique().tolist())
invalid_machine_type_count = int((~raw_df["Type"].astype(str).str.strip().isin(["L", "M", "H"])).sum())
type_validation_rows.append(
    {
        "column": "Type",
        "expected_type": "categorical: L, M, H",
        "invalid_count": invalid_machine_type_count,
        "validation_status": "Pass" if invalid_machine_type_count == 0 else "Fail",
    }
)

type_validation_summary = pd.DataFrame(type_validation_rows)
display(type_validation_summary)
print(f"Observed machine type values: {machine_type_values}")

In [ ]:
clean_df = raw_df.rename(columns=COLUMN_RENAME_MAP).copy()

clean_df["product_id"] = clean_df["product_id"].astype(str).str.strip()
clean_df["machine_type"] = clean_df["machine_type"].astype(str).str.strip()

for column in INTEGER_CLEAN_COLUMNS + FLOAT_CLEAN_COLUMNS:
    clean_df[column] = pd.to_numeric(clean_df[column], errors="coerce")

if clean_df[INTEGER_CLEAN_COLUMNS + FLOAT_CLEAN_COLUMNS].isna().any().any():
    invalid_numeric_summary = clean_df[INTEGER_CLEAN_COLUMNS + FLOAT_CLEAN_COLUMNS].isna().sum()
    raise ValueError(f"Numeric conversion failed: {invalid_numeric_summary[invalid_numeric_summary > 0].to_dict()}")

for column in INTEGER_CLEAN_COLUMNS:
    non_integer_count = int(((clean_df[column] % 1) != 0).sum())
    if non_integer_count > 0:
        raise ValueError(f"Column {column} contains {non_integer_count} non-integer values.")
    clean_df[column] = clean_df[column].astype("int64")

for column in FLOAT_CLEAN_COLUMNS:
    clean_df[column] = clean_df[column].astype("float64")

binary_validation = []
for column in BINARY_CLEAN_COLUMNS:
    invalid_values = sorted(set(clean_df.loc[~clean_df[column].isin([0, 1]), column].tolist()))
    binary_validation.append(
        {
            "column": column,
            "invalid_values": invalid_values,
            "validation_status": "Pass" if not invalid_values else "Fail",
        }
    )

display(pd.DataFrame(binary_validation))

if type_validation_summary["validation_status"].eq("Fail").any() or any(row["invalid_values"] for row in binary_validation):
    record_action(
        "Data Type Validation",
        "Invalid data types or indicator values detected",
        "Stop the process and correct source data before loading clean tables.",
        "Numeric sensor fields, categorical machine type, and binary failure indicators are required for valid SQL KPIs.",
        "Prevents broken calculations, invalid model training data, and misleading maintenance dashboards.",
    )
    display(action_table("Data Type Validation"))
    raise ValueError("Data type validation failed.")

record_action(
    "Data Type Validation",
    "All required fields passed type validation",
    "Convert fields to production-ready clean data types.",
    "Validated numeric fields can support SQL aggregation, engineered features, and model training.",
    "Creates a trusted typed dataset for maintenance KPIs and Power BI reporting.",
)

display(clean_df.dtypes.to_frame("clean_dtype"))
display(action_table("Data Type Validation"))

## Section 6: Outlier Analysis

Outliers in industrial data can represent true extreme operating conditions, sensor errors, or rare failure-driving behavior. This notebook does **not** automatically remove outliers. Instead, it identifies them and documents the decision to preserve them unless domain review proves they are invalid.

In [ ]:
def iqr_outlier_summary(dataframe: pd.DataFrame, columns: list[str]) -> pd.DataFrame:
    rows = []
    for column in columns:
        q1 = dataframe[column].quantile(0.25)
        q3 = dataframe[column].quantile(0.75)
        iqr = q3 - q1
        lower_bound = q1 - 1.5 * iqr
        upper_bound = q3 + 1.5 * iqr
        outlier_count = int(((dataframe[column] < lower_bound) | (dataframe[column] > upper_bound)).sum())
        rows.append(
            {
                "field": column,
                "q1": round(q1, 4),
                "q3": round(q3, 4),
                "iqr": round(iqr, 4),
                "lower_bound": round(lower_bound, 4),
                "upper_bound": round(upper_bound, 4),
                "outlier_count": outlier_count,
                "outlier_percent": round(100 * outlier_count / len(dataframe), 4),
            }
        )
    return pd.DataFrame(rows)

outlier_summary = iqr_outlier_summary(clean_df, SENSOR_COLUMNS)
display(outlier_summary)

In [ ]:
fig, axes = plt.subplots(len(SENSOR_COLUMNS), 2, figsize=(14, 18))

for row_index, column in enumerate(SENSOR_COLUMNS):
    sns.boxplot(x=clean_df[column], ax=axes[row_index, 0], color="#4C78A8")
    axes[row_index, 0].set_title(f"Boxplot: {column}")
    axes[row_index, 0].set_xlabel(column)

    sns.histplot(clean_df[column], kde=True, ax=axes[row_index, 1], color="#F58518")
    axes[row_index, 1].set_title(f"Distribution: {column}")
    axes[row_index, 1].set_xlabel(column)

plt.tight_layout()
plt.show()

In [ ]:
for row in outlier_summary.to_dict("records"):
    record_action(
        "Outlier Analysis",
        f"{row['field']} has {row['outlier_count']} IQR outlier(s)",
        "Do not automatically remove outliers.",
        "Industrial extremes can represent real stress conditions and may be important predictors of failure.",
        "Preserves critical high-risk operating behavior for reliability analysis, feature engineering, and predictive maintenance modeling.",
    )

display(action_table("Outlier Analysis"))

## Section 7: Feature Consistency Checks

This section checks invalid values, negative values, impossible measurements, and inconsistent failure labels. Findings are documented before feature engineering.

In [ ]:
failure_mode_columns = ["twf", "hdf", "pwf", "osf", "rnf"]
failure_mode_sum = clean_df[failure_mode_columns].sum(axis=1)

consistency_checks = pd.DataFrame(
    [
        {"check": "Blank product_id", "issue_count": int((clean_df["product_id"].str.len() == 0).sum())},
        {"check": "Invalid machine_type", "issue_count": int((~clean_df["machine_type"].isin(["L", "M", "H"])).sum())},
        {"check": "Non-positive air_temperature_k", "issue_count": int((clean_df["air_temperature_k"] <= 0).sum())},
        {"check": "Non-positive process_temperature_k", "issue_count": int((clean_df["process_temperature_k"] <= 0).sum())},
        {"check": "Non-positive rotational_speed_rpm", "issue_count": int((clean_df["rotational_speed_rpm"] <= 0).sum())},
        {"check": "Negative torque_nm", "issue_count": int((clean_df["torque_nm"] < 0).sum())},
        {"check": "Negative tool_wear_min", "issue_count": int((clean_df["tool_wear_min"] < 0).sum())},
        {"check": "machine_failure=1 with no failure mode", "issue_count": int(((clean_df["machine_failure"] == 1) & (failure_mode_sum == 0)).sum())},
        {"check": "failure mode active while machine_failure=0", "issue_count": int(((clean_df["machine_failure"] == 0) & (failure_mode_sum > 0)).sum())},
    ]
)

display(consistency_checks)

In [ ]:
blocking_checks = consistency_checks.loc[
    consistency_checks["check"].isin(
        [
            "Blank product_id",
            "Invalid machine_type",
            "Non-positive air_temperature_k",
            "Non-positive process_temperature_k",
            "Non-positive rotational_speed_rpm",
            "Negative torque_nm",
            "Negative tool_wear_min",
        ]
    )
    & (consistency_checks["issue_count"] > 0)
]

label_warning_checks = consistency_checks.loc[
    consistency_checks["check"].isin(
        [
            "machine_failure=1 with no failure mode",
            "failure mode active while machine_failure=0",
        ]
    )
    & (consistency_checks["issue_count"] > 0)
]

if blocking_checks.empty:
    record_action(
        "Feature Consistency Checks",
        "No invalid, negative, or impossible sensor measurements detected",
        "Proceed with feature engineering.",
        "Core operating values are within valid physical and business-rule ranges.",
        "Supports reliable stress scoring, KPI calculation, and dashboard interpretation.",
    )
else:
    record_action(
        "Feature Consistency Checks",
        "Invalid or impossible measurement values detected",
        "Stop the process and correct source records before database loading.",
        "Physically impossible operating values would corrupt maintenance features and risk rankings.",
        "Prevents false maintenance recommendations and protects KPI credibility.",
    )

if label_warning_checks.empty:
    record_action(
        "Feature Consistency Checks",
        "No aggregate/detail failure-label inconsistency detected",
        "Use failure labels as provided.",
        "The aggregate target and detailed failure modes align.",
        "Improves confidence in failure-category reporting and model labels.",
    )
else:
    record_action(
        "Feature Consistency Checks",
        "Aggregate/detail failure-label inconsistency detected",
        "Preserve original labels and document the inconsistency instead of rewriting labels.",
        "AI4I can contain label edge cases; rewriting labels without domain confirmation may change the modeling target.",
        "Maintains auditability while making label limitations transparent for stakeholders.",
    )

display(action_table("Feature Consistency Checks"))

if not blocking_checks.empty:
    raise ValueError("Blocking feature consistency issues must be resolved before continuing.")

## Section 8: Feature Engineering

The features below translate raw operating signals into maintenance intelligence fields that are easier to analyze in SQL, Power BI, and machine learning workflows.

| Feature | Business Purpose | Formula | Expected Value |
|---|---|---|---|
| Temperature Difference | Measures thermal gap between process and surrounding air. | `process_temperature_k - air_temperature_k` | Positive numeric value in Kelvin. |
| Mechanical Power | Estimates mechanical load from torque and speed. | `torque_nm x rotational_speed_rpm x 2 x pi / 60` | Positive numeric value in watts. |
| Tool Wear Risk Category | Segments equipment by tool wear severity. | Low: `<120`, Medium: `120-199`, High: `>=200` minutes | `Low`, `Medium`, or `High`. |
| Operating Stress Score | Combines normalized thermal, torque, tool wear, and power stress. | Average of min-max scaled stress inputs x 100 | Score from 0 to 100. |
| Failure Count | Counts active detailed failure-mode flags. | `TWF + HDF + PWF + OSF + RNF` | Integer from 0 to 5. |
| Primary Failure Mode | Assigns the first active failure category for reporting. | Priority order: TWF, HDF, PWF, OSF, RNF | Failure label or `No Failure`. |

In [ ]:
def classify_tool_wear_risk(tool_wear_min: int) -> str:
    if tool_wear_min >= 200:
        return "High"
    if tool_wear_min >= 120:
        return "Medium"
    return "Low"

def resolve_primary_failure_mode(row: pd.Series) -> str:
    failure_modes = [
        ("twf", "Tool Wear Failure"),
        ("hdf", "Heat Dissipation Failure"),
        ("pwf", "Power Failure"),
        ("osf", "Overstrain Failure"),
        ("rnf", "Random Failure"),
    ]
    for column, label in failure_modes:
        if int(row[column]) == 1:
            return label
    return "No Failure"

def min_max_scale(series: pd.Series) -> pd.Series:
    minimum = series.min()
    maximum = series.max()
    if maximum == minimum:
        return pd.Series(0.0, index=series.index)
    return (series - minimum) / (maximum - minimum)

analytics_df = clean_df.copy()
analytics_df["temperature_difference_k"] = (
    analytics_df["process_temperature_k"] - analytics_df["air_temperature_k"]
).round(4)
analytics_df["mechanical_power_watts"] = (
    analytics_df["torque_nm"] * analytics_df["rotational_speed_rpm"] * (2 * math.pi / 60)
).round(4)
analytics_df["tool_wear_risk_level"] = analytics_df["tool_wear_min"].apply(classify_tool_wear_risk)
analytics_df["failure_mode_count"] = analytics_df[failure_mode_columns].sum(axis=1).astype("int64")
analytics_df["primary_failure_mode"] = analytics_df.apply(resolve_primary_failure_mode, axis=1)

stress_inputs = pd.DataFrame(
    {
        "temperature_difference_k": analytics_df["temperature_difference_k"],
        "torque_nm": analytics_df["torque_nm"],
        "tool_wear_min": analytics_df["tool_wear_min"],
        "mechanical_power_watts": analytics_df["mechanical_power_watts"],
    }
)
analytics_df["operating_stress_score"] = (
    pd.concat([min_max_scale(stress_inputs[column]) for column in stress_inputs.columns], axis=1).mean(axis=1) * 100
).round(4)

feature_preview_columns = [
    "observation_id",
    "product_id",
    "machine_type",
    "temperature_difference_k",
    "mechanical_power_watts",
    "tool_wear_risk_level",
    "operating_stress_score",
    "failure_mode_count",
    "primary_failure_mode",
]
display(analytics_df[feature_preview_columns].head())

In [ ]:
feature_summary = pd.DataFrame(
    [
        {
            "feature": "temperature_difference_k",
            "min": analytics_df["temperature_difference_k"].min(),
            "max": analytics_df["temperature_difference_k"].max(),
            "null_count": int(analytics_df["temperature_difference_k"].isna().sum()),
        },
        {
            "feature": "mechanical_power_watts",
            "min": analytics_df["mechanical_power_watts"].min(),
            "max": analytics_df["mechanical_power_watts"].max(),
            "null_count": int(analytics_df["mechanical_power_watts"].isna().sum()),
        },
        {
            "feature": "operating_stress_score",
            "min": analytics_df["operating_stress_score"].min(),
            "max": analytics_df["operating_stress_score"].max(),
            "null_count": int(analytics_df["operating_stress_score"].isna().sum()),
        },
    ]
)

display(feature_summary)
display(analytics_df["tool_wear_risk_level"].value_counts().rename_axis("risk_level").reset_index(name="count"))
display(analytics_df["primary_failure_mode"].value_counts().rename_axis("failure_mode").reset_index(name="count"))

record_action(
    "Feature Engineering",
    "Maintenance intelligence features created",
    "Create derived features and keep source measurements unchanged.",
    "Derived features improve interpretability while preserving auditability of original operating data.",
    "Supports Power BI segmentation, SQL KPI analysis, and predictive maintenance model preparation.",
)
display(action_table("Feature Engineering"))

## Section 9: Clean Dataset Creation

This section creates the final cleaned dataset and stores it in SQLite tables:

- `machine_clean`: validated and typed machine observations
- `machine_analytics`: engineered maintenance analytics dataset

The database load uses the existing DDL and a transaction so partial loads are not committed.

In [ ]:
source_file = DATA_PATH.name
ingestion_timestamp = datetime.now(timezone.utc).isoformat()

machine_clean = clean_df.copy()
machine_clean["source_file"] = source_file
machine_clean["ingestion_timestamp"] = ingestion_timestamp
machine_clean = machine_clean[
    [
        "observation_id",
        "product_id",
        "machine_type",
        "air_temperature_k",
        "process_temperature_k",
        "rotational_speed_rpm",
        "torque_nm",
        "tool_wear_min",
        "machine_failure",
        "twf",
        "hdf",
        "pwf",
        "osf",
        "rnf",
        "source_file",
        "ingestion_timestamp",
    ]
]

machine_analytics = analytics_df.copy()
machine_analytics["source_file"] = source_file
machine_analytics["ingestion_timestamp"] = ingestion_timestamp
machine_analytics = machine_analytics[
    [
        "observation_id",
        "product_id",
        "machine_type",
        "air_temperature_k",
        "process_temperature_k",
        "temperature_difference_k",
        "rotational_speed_rpm",
        "torque_nm",
        "mechanical_power_watts",
        "tool_wear_min",
        "tool_wear_risk_level",
        "operating_stress_score",
        "machine_failure",
        "failure_mode_count",
        "primary_failure_mode",
        "twf",
        "hdf",
        "pwf",
        "osf",
        "rnf",
        "source_file",
        "ingestion_timestamp",
    ]
]

print(f"machine_clean rows: {len(machine_clean):,}")
print(f"machine_analytics rows: {len(machine_analytics):,}")

In [ ]:
def initialize_database(database_path: Path, ddl_path: Path) -> None:
    database_path.parent.mkdir(parents=True, exist_ok=True)
    ddl_sql = ddl_path.read_text(encoding="utf-8")
    with sqlite3.connect(database_path) as connection:
        connection.execute("PRAGMA foreign_keys = ON;")
        connection.executescript(ddl_sql)

def load_clean_tables(database_path: Path, clean_table: pd.DataFrame, analytics_table: pd.DataFrame) -> None:
    with sqlite3.connect(database_path) as connection:
        connection.execute("PRAGMA foreign_keys = ON;")
        try:
            connection.execute("BEGIN;")
            connection.execute("DELETE FROM machine_analytics;")
            connection.execute("DELETE FROM machine_clean;")
            clean_table.to_sql("machine_clean", connection, if_exists="append", index=False, method="multi", chunksize=1000)
            analytics_table.to_sql("machine_analytics", connection, if_exists="append", index=False, method="multi", chunksize=1000)
            connection.commit()
        except Exception:
            connection.rollback()
            raise

initialize_database(DATABASE_PATH, DDL_PATH)
load_clean_tables(DATABASE_PATH, machine_clean, machine_analytics)

with sqlite3.connect(DATABASE_PATH) as connection:
    loaded_counts = pd.DataFrame(
        [
            {"table": "machine_clean", "row_count": connection.execute("SELECT COUNT(*) FROM machine_clean").fetchone()[0]},
            {"table": "machine_analytics", "row_count": connection.execute("SELECT COUNT(*) FROM machine_analytics").fetchone()[0]},
        ]
    )

display(loaded_counts)

record_action(
    "Clean Dataset Creation",
    "Clean and analytics datasets loaded to SQLite",
    "Replace existing machine_clean and machine_analytics records using a database transaction.",
    "A full refresh prevents stale analytical records and ensures clean and analytics tables stay aligned.",
    "Provides reliable SQL and Power BI datasets for KPI reporting and maintenance intelligence.",
)
display(action_table("Clean Dataset Creation"))

## Section 10: Data Quality Summary

This final section summarizes issues found, actions taken, and the business impact of the cleaning process.

In [ ]:
data_quality_summary = pd.DataFrame(quality_actions)
display(data_quality_summary)

issue_summary = data_quality_summary.groupby("Section", as_index=False).agg(
    issues_documented=("Issue", "count"),
    decisions=("Decision", lambda values: " | ".join(sorted(set(values)))),
)
display(issue_summary)

### Final Business Impact

The cleaned dataset improves maintenance analytics by ensuring that:

- Failure rates are calculated from complete and non-duplicated records.
- Sensor values are valid and typed for SQL aggregation.
- Operational outliers are retained for reliability and stress analysis.
- Failure label limitations are documented instead of hidden.
- Engineered features are explainable and aligned with maintenance decision making.
- `machine_clean` and `machine_analytics` are ready for SQL analytics, KPI reporting, Power BI, and predictive modeling.